# Get Data

## Find files

In [ ]:
import os.path

def find_files(*, base_path: str, pattern: str) -> list[str]:
    """Busca recursivamente arquivos contendo 'pattern' no nome."""
    return [
        os.path.join(root, file)
        for root, _, files in os.walk(base_path)
        for file in files
        if pattern in file
    ]


## Extract/Transform

In [ ]:
import pandas as pd


def _standard_hour(input_hour: int, /) -> str:
    hour = str(input_hour).replace("UTC", "").strip()

    if ':' in hour:
        hour = hour.replace(":", "")
    
    return hour

def extract_and_transform(files: list[str], /) -> pd.DataFrame:
    """Carrega CSVs do INMET e aplica renomeação + limpeza inicial."""
    df_list = []
    for file in files:
        df = pd.read_csv(file, encoding="ISO-8859-1", sep=";", skiprows=8)
        df = df.rename(
            columns={
                "DATA (YYYY-MM-DD)": "Data",
                "HORA (UTC)": "Hora UTC",
                "RADIACAO GLOBAL (KJ/m²)": "RADIACAO GLOBAL (Kj/m²)",
            }
        )
        df_list.append(df)

    df = pd.concat(df_list, ignore_index=True).drop(columns=["Unnamed: 19"])

    # Converter datas
    df["Data"] = pd.to_datetime(df["Data"].str.replace("-", "/"))
    df["Hora UTC"] = pd.to_datetime(
        df["Hora UTC"].apply(_standard_hour), format="%H%M"
    ).dt.time

    # Corrigir colunas numéricas (vírgula -> ponto)
    cols_to_float = [
        "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
        "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
        "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
        "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
        "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
        "RADIACAO GLOBAL (Kj/m²)",
        "TEMPERATURA DO PONTO DE ORVALHO (°C)",
        "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
        "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
        "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
        "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
        "VENTO, VELOCIDADE HORARIA (m/s)",
    ]
    for col in cols_to_float:
        df[col] = df[col].str.replace(",", ".").astype(float)

    # Rajada de vento já vem como float
    df["VENTO, RAJADA MAXIMA (m/s)"] = df[
        "VENTO, DIREÇÃO HORARIA (gr) (° (gr))"
    ].astype(float)

    return df


## Remove outliers

In [ ]:
from datetime import time

import pandas as pd


def remove_outliers(df: pd.DataFrame, /) -> pd.DataFrame:
    """Remove registros sem radiação solar entre 6h e 20h."""
    inicio, fim = time(6, 0), time(20, 0)
    mask = (
        df["RADIACAO GLOBAL (Kj/m²)"].isnull()
        & (df["Hora UTC"] >= inicio)
        & (df["Hora UTC"] < fim)
    )
    return df[~mask]

## Engineer features

In [ ]:
import pandas as pd

def engineer_features(df: pd.DataFrame, /) -> pd.DataFrame:
    """Cria datetime, year, month, day, hour e remove colunas irrelevantes."""
    df["datetime"] = pd.to_datetime(
        df["Data"].astype(str) + " " + df["Hora UTC"].astype(str)
    )
    df["RADIACAO GLOBAL (Kj/m²)"] = df["RADIACAO GLOBAL (Kj/m²)"].fillna(0)

    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["day"] = df["datetime"].dt.day
    df["hour"] = df["datetime"].dt.hour

    drop_cols = [
        "Data",
        "Hora UTC",
        "datetime",
        "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
        "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
        "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
        "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
        "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
        "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    ]
    return df.drop(columns=[c for c in drop_cols if c in df.columns]).dropna()


## Put all together

In [ ]:
from configs import CONFIGS

import pandas as pd

def get_sample_data() -> pd.DataFrame:
    base_path = CONFIGS.DATA_CSV_FOLDER
    pattern = "INMET_SE_SP_A705_BAURU"

    files = find_files(base_path=base_path, pattern=pattern)
    df = extract_and_transform(files)
    # df = remove_outliers(df)
    # df = engineer_features(df)

    return df


# df = get_sample_data()

# Descrição de colunas




| Coluna | Descrição |
|--------|-----------|
| **PRECIPITAÇÃO TOTAL, HORÁRIO (mm)** | Quantidade total de chuva acumulada em um intervalo de uma hora, medida em milímetros (mm). |
| **PRESSÃO ATMOSFÉRICA AO NÍVEL DA ESTAÇÃO, HORÁRIA (mB)** | Pressão atmosférica medida no local da estação meteorológica em milibares (mB). |
| **PRESSÃO ATMOSFÉRICA NA HORA ANT. (AUT) (mB)** | Valor da pressão atmosférica registrado na hora anterior. |
| **RADIAÇÃO GLOBAL (Kj/m²)** | Quantidade total de energia solar recebida por metro quadrado de superfície em uma hora, medida em kilojoules por metro quadrado (Kj/m²). |
| **TEMPERATURA DO AR - BULBO SECO, HORÁRIA (°C)** | Temperatura do ar medida por um termômetro comum (sem influência da umidade), expressa em graus Celsius (°C). |
| **TEMPERATURA DO PONTO DE ORVALHO (°C)** | Temperatura na qual o ar se torna saturado de umidade e ocorre a condensação, formando orvalho. |
| **TEMPERATURA NA HORA ANT. (AUT) (°C)** | Temperatura do ar registrada na hora anterior. |
| **TEMPERATURA ORVALHO NA HORA ANT. (AUT) (°C)** | Temperatura do ponto de orvalho registrada na hora anterior. |
| **UMIDADE REL. NA HORA ANT. (AUT) (%)** | Umidade relativa do ar registrada na hora anterior. |
| **UMIDADE RELATIVA DO AR, HORÁRIA (%)** | Quantidade de vapor d'água presente no ar em relação à quantidade máxima que ele pode conter a uma determinada temperatura, expressa em porcentagem. |
| **VENTO, DIREÇÃO HORÁRIA (gr) (° (gr))** | Direção média do vento ao longo da última hora, medida em graus (°) em relação ao norte. |
| **VENTO, RAJADA MÁXIMA (m/s)** | Maior velocidade do vento em um curto intervalo de tempo dentro da última hora, medida em metros por segundo (m/s). |
| **VENTO, VELOCIDADE HORÁRIA (m/s)** | Velocidade média do vento ao longo da última hora, medida em metros por segundo (m/s). |



# Graphs

## Analisando o porque `RADIACAO GLOBAL` tem um padrão de valores nulos

TODO(hspadim1): Verificar se o site deles dão uma explicação melhor



In [ ]:
import matplotlib.pyplot as plt

df = get_sample_data()
radiacao_nan_df = df[df['RADIACAO GLOBAL (Kj/m²)'].isnull()]
na_por_hora = radiacao_nan_df['Hora UTC'].value_counts().sort_index()

plt.figure(figsize=(12, 6))
na_por_hora.plot(kind='bar', color='orange')
plt.title('Frequência de Nulls por horário na radiação solar')
plt.xlabel('Horário')
plt.ylabel('Número de Nulls')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
# plt.show()
plt.savefig("fig1.png")